# Zomato Analytics — Data Quality Checks

**Purpose**: Run rule-based DQ checks on every Bronze table. Records that pass
all critical checks are written to `bronze.<table>_validated` and allowed to
proceed to Silver. Records that fail any critical check are written to
`bronze.<table>_quarantine` for investigation.

**Gate**: If the pass rate for any table drops below the configured threshold
(default 90%), the notebook raises an exception and halts the pipeline —
Silver transformation will not run.

**Checks performed per table**:

| Table | Checks |
|---|---|
| customers | NOT NULL IDs/email, email format, lat/lon range, avg_order_value ≥ 0 |
| restaurants | NOT NULL IDs/name, rating 1–5, avg_cost_for_two > 0, lat/lon range |
| orders | NOT NULL IDs, total_amount ≥ 0, num_items ≥ 1, valid order_status |
| deliveries | NOT NULL IDs, delivery_time_mins > 0, distance > 0, rating 1–5 or null |
| reviews | NOT NULL IDs, rating 1–5 |

**Outputs**:
- `{CATALOG}.bronze.{table}_validated` — records that passed all checks
- `{CATALOG}.bronze.{table}_quarantine` — records that failed one or more checks
- Audit entry written to `{CATALOG}.audit.pipeline_audit_log`

## 1. Configuration

In [0]:
dbutils.widgets.text("catalog_name", "zomato_analytics", "Unity Catalog name")
dbutils.widgets.text("env", "dev", "Environment")
dbutils.widgets.text("pass_rate_threshold", "90.0", "Min pass rate % before blocking pipeline")

CATALOG = dbutils.widgets.get("catalog_name")
ENV = dbutils.widgets.get("env")
PASS_RATE_THRESHOLD = float(dbutils.widgets.get("pass_rate_threshold"))

BRONZE_FQN = f"{CATALOG}.bronze"

print(f"Catalog            : {CATALOG}")
print(f"Bronze namespace   : {BRONZE_FQN}")
print(f"Pass rate gate     : {PASS_RATE_THRESHOLD}%")
print(f"Env                : {ENV}")



##Import Audit class

In [0]:
%run /Workspace/Repos/bhamidipati.suryans@gmail.com/zomato-databricks-pipeline/logger/audit_table

## 2. DQ Framework

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, lit, regexp_extract, length, trim,
    current_timestamp, concat_ws,
)
from dataclasses import dataclass, field
from typing import List, Callable


@dataclass
class DQRule:
    """
    A single data quality rule.

    Attributes:
        name        : Short identifier shown in the DQ report.
        description : Human-readable description of what is being checked.
        condition   : A Spark Column expression that evaluates to True when
                      the record PASSES the check and False when it fails.
        severity    : 'critical'  — failing records are quarantined and
                                    excluded from downstream processing.
                      'warning'   — failing records are flagged but still
                                    allowed to proceed.
    """
    name: str
    description: str
    condition: object          # pyspark Column
    severity: str = "critical" # 'critical' | 'warning'


@dataclass
class DQResult:
    """Holds the outcome of running all rules against one table."""
    table: str
    total_records: int
    passed_records: int
    quarantined_records: int
    warned_records: int
    pass_rate_pct: float
    rule_summary: List[dict] = field(default_factory=list)
    blocked: bool = False      # True when pass_rate < threshold


def run_dq_checks(
    df: DataFrame,
    rules: List[DQRule],
    table_name: str,
    validated_target: str,
    quarantine_target: str,
    threshold_pct: float = 90.0,
) -> DQResult:
    """
    Apply DQ rules to a DataFrame.

    - Records passing ALL critical rules  → written to validated_target
    - Records failing ANY critical rule   → written to quarantine_target
      (with _dq_failed_rules column listing which rules failed)
    - Warning-only failures do not quarantine the record but are flagged.

    Returns a DQResult summary.
    """
    total = df.count()
    df_work = df

    # Build composite pass/fail columns for each rule
    rule_summary = []
    critical_pass_cols = []

    for rule in rules:
        pass_col = f"_dq_pass_{rule.name}"
        df_work = df_work.withColumn(pass_col, rule.condition)
        fail_count = df_work.filter(~col(pass_col)).count()
        pass_count = total - fail_count
        rule_summary.append({
            "rule": rule.name,
            "description": rule.description,
            "severity": rule.severity,
            "passed": pass_count,
            "failed": fail_count,
            "pass_rate_pct": round(pass_count / max(total, 1) * 100, 2),
        })
        if rule.severity == "critical":
            critical_pass_cols.append(pass_col)

    # A record passes overall if it passes every critical rule
    if critical_pass_cols:
        from pyspark.sql.functions import array, array_contains
        overall_pass = critical_pass_cols[0]
        for c in critical_pass_cols[1:]:
            overall_pass = f"{overall_pass} AND {c}"
        # Build as a Column expression
        from functools import reduce
        from pyspark.sql import functions as F
        pass_expr = reduce(lambda a, b: a & col(b), critical_pass_cols[1:], col(critical_pass_cols[0]))
    else:
        from pyspark.sql.functions import lit as _lit
        pass_expr = _lit(True)

    df_work = df_work.withColumn("_dq_passed", pass_expr)

    # Tag quarantined records with which rules they failed
    failed_rule_cols = [col(f"_dq_pass_{r.name}") for r in rules if r.severity == "critical"]
    failed_rule_names = [r.name for r in rules if r.severity == "critical"]

    def build_failed_rules_col():
        parts = []
        for r in rules:
            if r.severity == "critical":
                parts.append(
                    when(~col(f"_dq_pass_{r.name}"), lit(r.name)).otherwise(lit(None))
                )
        if not parts:
            return lit(None).cast("string")
        return concat_ws(", ", *parts)

    df_work = df_work.withColumn("_dq_failed_rules", build_failed_rules_col())
    df_work = df_work.withColumn("_dq_checked_at", current_timestamp())

    # Drop helper columns before writing
    helper_cols = [f"_dq_pass_{r.name}" for r in rules]

    validated_df = (
        df_work.filter(col("_dq_passed"))
        .drop(*helper_cols, "_dq_passed", "_dq_failed_rules")
    )
    quarantine_df = (
        df_work.filter(~col("_dq_passed"))
        .drop(*helper_cols, "_dq_passed")
    )

    # Write outputs
    (
        validated_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(validated_target)
    )
    (
        quarantine_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(quarantine_target)
    )

    passed = validated_df.count()
    quarantined = quarantine_df.count()
    pass_rate = round(passed / max(total, 1) * 100, 2)
    blocked = pass_rate < threshold_pct

    return DQResult(
        table=table_name,
        total_records=total,
        passed_records=passed,
        quarantined_records=quarantined,
        warned_records=0,
        pass_rate_pct=pass_rate,
        rule_summary=rule_summary,
        blocked=blocked,
    )

## 3. DQ Rules Definition

In [0]:
from pyspark.sql.functions import col, regexp_extract, length, trim, lower

VALID_ORDER_STATUSES = ["Placed", "Confirmed", "Preparing", "Out for Delivery",
                        "Delivered", "Cancelled", "Refunded"]

DQ_RULES = {
    "customers": [
        DQRule(
            name="customer_id_not_null",
            description="customer_id must not be null or empty",
            condition=col("customer_id").isNotNull() & (trim(col("customer_id")) != ""),
        ),
        DQRule(
            name="email_not_null",
            description="email must not be null",
            condition=col("email").isNotNull(),
        ),
        DQRule(
            name="email_format",
            description="email must match standard format (x@x.x)",
            condition=col("email").rlike(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z]{2,}$"),
        ),
        DQRule(
            name="latitude_range",
            description="latitude must be between -90 and 90",
            condition=col("latitude").isNotNull() & col("latitude").between(-90.0, 90.0),
        ),
        DQRule(
            name="longitude_range",
            description="longitude must be between -180 and 180",
            condition=col("longitude").isNotNull() & col("longitude").between(-180.0, 180.0),
        ),
        DQRule(
            name="avg_order_value_non_negative",
            description="avg_order_value must be >= 0",
            condition=col("avg_order_value").isNull() | (col("avg_order_value") >= 0),
            severity="warning",
        ),
    ],

    "restaurants": [
        DQRule(
            name="restaurant_id_not_null",
            description="restaurant_id must not be null or empty",
            condition=col("restaurant_id").isNotNull() & (trim(col("restaurant_id")) != ""),
        ),
        DQRule(
            name="name_not_null",
            description="restaurant name must not be null",
            condition=col("name").isNotNull() & (trim(col("name")) != ""),
        ),
        DQRule(
            name="rating_range",
            description="rating must be between 1.0 and 5.0",
            condition=col("rating").isNull() | col("rating").between(1.0, 5.0),
            severity="warning",
        ),
        DQRule(
            name="avg_cost_positive",
            description="avg_cost_for_two must be > 0",
            condition=col("avg_cost_for_two").isNull() | (col("avg_cost_for_two") > 0),
            severity="warning",
        ),
        DQRule(
            name="latitude_range",
            description="latitude must be between -90 and 90",
            condition=col("latitude").isNotNull() & col("latitude").between(-90.0, 90.0),
        ),
        DQRule(
            name="longitude_range",
            description="longitude must be between -180 and 180",
            condition=col("longitude").isNotNull() & col("longitude").between(-180.0, 180.0),
        ),
    ],

    "orders": [
        DQRule(
            name="order_id_not_null",
            description="order_id must not be null",
            condition=col("order_id").isNotNull(),
        ),
        DQRule(
            name="customer_id_not_null",
            description="customer_id must not be null",
            condition=col("customer_id").isNotNull(),
        ),
        DQRule(
            name="restaurant_id_not_null",
            description="restaurant_id must not be null",
            condition=col("restaurant_id").isNotNull(),
        ),
        DQRule(
            name="total_amount_non_negative",
            description="total_amount must be >= 0",
            condition=col("total_amount").isNull() | (col("total_amount") >= 0),
        ),
        DQRule(
            name="num_items_positive",
            description="num_items must be >= 1",
            condition=col("num_items").isNull() | (col("num_items") >= 1),
        ),
        DQRule(
            name="valid_order_status",
            description=f"order_status must be one of {VALID_ORDER_STATUSES}",
            condition=col("order_status").isin(VALID_ORDER_STATUSES),
            severity="warning",
        ),
    ],

    "deliveries": [
        DQRule(
            name="delivery_id_not_null",
            description="delivery_id must not be null",
            condition=col("delivery_id").isNotNull(),
        ),
        DQRule(
            name="order_id_not_null",
            description="order_id must not be null",
            condition=col("order_id").isNotNull(),
        ),
        DQRule(
            name="delivery_time_positive",
            description="delivery_time_mins must be > 0",
            condition=col("delivery_time_mins").isNull() | (col("delivery_time_mins") > 0),
        ),
        DQRule(
            name="delivery_distance_positive",
            description="delivery_distance_km must be > 0",
            condition=col("delivery_distance_km").isNull() | (col("delivery_distance_km") > 0),
        ),
        DQRule(
            name="delivery_rating_range",
            description="delivery_rating must be 1–5 or null",
            condition=col("delivery_rating").isNull() | col("delivery_rating").between(1, 5),
            severity="warning",
        ),
    ],

    "reviews": [
        DQRule(
            name="review_id_not_null",
            description="review_id must not be null",
            condition=col("review_id").isNotNull(),
        ),
        DQRule(
            name="order_id_not_null",
            description="order_id must not be null",
            condition=col("order_id").isNotNull(),
        ),
        DQRule(
            name="customer_id_not_null",
            description="customer_id must not be null",
            condition=col("customer_id").isNotNull(),
        ),
        DQRule(
            name="restaurant_id_not_null",
            description="restaurant_id must not be null",
            condition=col("restaurant_id").isNotNull(),
        ),
        DQRule(
            name="rating_range",
            description="rating must be between 1 and 5",
            condition=col("rating").isNotNull() & col("rating").between(1, 5),
        ),
    ],
}

## 4. Run DQ Checks on All Bronze Tables

In [0]:
import json

TABLES = ["customers", "restaurants", "orders", "deliveries", "reviews"]

dq_results = []

print("=" * 70)
print("  DATA QUALITY CHECKS — BRONZE LAYER")
print("=" * 70)
from datetime import date, datetime
from pyspark.sql import Row
audit_table=audit()


try:
    _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    _workflow_name = _ctx.jobName().getOrElse("manual_run")
except Exception:
    _workflow_name = "manual_run"


_load_date = date.today()
_run_dt = datetime.utcnow()

for table in TABLES:
    print(f"\n{'─' * 50}")
    print(f"  Table: {BRONZE_FQN}.{table}")
    print(f"{'─' * 50}")

    df = spark.table(f"{BRONZE_FQN}.{table}")
    rules = DQ_RULES.get(table, [])
    validated_target = f"{BRONZE_FQN}.{table}_validated"
    quarantine_target = f"{BRONZE_FQN}.{table}_quarantine"

    result = run_dq_checks(
        df=df,
        rules=rules,
        table_name=table,
        validated_target=validated_target,
        quarantine_target=quarantine_target,
        threshold_pct=PASS_RATE_THRESHOLD,
    )
    dq_results.append(result)
 
    # Print rule-level summary
    for rs in result.rule_summary:
        status = "PASS" if rs["failed"] == 0 else ("WARN" if rs["severity"] == "warning" else "FAIL")
        print(f"  [{status}] {rs['rule']}: {rs['passed']:,} passed / {rs['failed']:,} failed  ({rs['pass_rate_pct']}%) — {rs['description']}")

    audit_table.insert_audit_information(
        run_datetime=_run_dt,
        workflow_name=_workflow_name,
        task_name="dq_checks",
        layer="quality",
        table_name=table,
        total_records=result.total_records,
        load_date=_load_date,
        status=status,
        error_message=(
            f"Pass rate {result.pass_rate_pct}% below threshold {PASS_RATE_THRESHOLD}% — "
            f"{result.quarantined_records:,} records quarantined"))
    
        
    print(f"✓ DQ audit entries written → {CATALOG}.audit.pipeline_audit_log")
    gate_status = "BLOCKED" if result.blocked else "OK"
    print(f"\n  Overall  : {result.passed_records:,} validated / {result.quarantined_records:,} quarantined")
    print(f"  Pass rate: {result.pass_rate_pct}%  (threshold: {PASS_RATE_THRESHOLD}%)  →  {gate_status}")
    print(f"  Validated table  : {validated_target}")
    print(f"  Quarantine table : {quarantine_target}")

print("\n" + "=" * 70)

## 5. DQ Summary Report

In [0]:
print("\n" + "=" * 70)
print("  DQ SUMMARY")
print("=" * 70)
print(f"  {'Table':<20} {'Total':>10} {'Validated':>10} {'Quarantined':>12} {'Pass %':>8} {'Gate':>8}")
print("  " + "-" * 68)

blocked_tables = []
for r in dq_results:
    gate = "BLOCKED" if r.blocked else "OK"
    print(f"  {r.table:<20} {r.total_records:>10,} {r.passed_records:>10,} {r.quarantined_records:>12,} {r.pass_rate_pct:>7}% {gate:>8}")
    if r.blocked:
        blocked_tables.append(r.table)

print("=" * 70)

total_in  = sum(r.total_records     for r in dq_results)
total_out = sum(r.passed_records    for r in dq_results)
total_qrt = sum(r.quarantined_records for r in dq_results)

print(f"\n  Total input records   : {total_in:,}")
print(f"  Total validated       : {total_out:,}")
print(f"  Total quarantined     : {total_qrt:,}")
print(f"  Overall pass rate     : {round(total_out / max(total_in, 1) * 100, 2)}%")

## 6. Write DQ Results to Audit Log

In [0]:
from datetime import date, datetime
from pyspark.sql import Row
audit_table=audit()


try:
    _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    _workflow_name = _ctx.jobName().getOrElse("manual_run")
except Exception:
    _workflow_name = "manual_run"


_load_date = date.today()
_run_dt = datetime.utcnow()

# audit_rows = [
#     Row(
#         run_datetime=_run_dt,
#         workflow_name=_workflow_name,
#         task_name="dq_checks",
#         layer="quality",
#         table_name=r.table,
#         total_records=r.total_records,
#         load_date=_load_date,
#         status="FAILED" if r.blocked else "SUCCESS",
#         error_message=(
#             f"Pass rate {r.pass_rate_pct}% below threshold {PASS_RATE_THRESHOLD}% — "
#             f"{r.quarantined_records:,} records quarantined"
#         ) if r.blocked else None,
#     )
#     for r in dq_results
# ]

# (
#     spark.createDataFrame(audit_rows,schema=sc)
#     .write.format("delta")
#     .mode("append")
#     .saveAsTable(f"{CATALOG}.audit.pipeline_audit_log")
# )


## 7. Pipeline Gate — Block if Any Table Below Threshold

In [0]:
if blocked_tables:
    raise Exception(
        f"DQ gate FAILED for {len(blocked_tables)} table(s): {blocked_tables}. "
        f"Pass rate dropped below {PASS_RATE_THRESHOLD}%. "
        f"Silver transformation is blocked until data quality issues are resolved. "
        f"Inspect {BRONZE_FQN}.<table>_quarantine for failing records."
    )

print("✅ All DQ checks passed — pipeline gate OPEN")
print(f"   Silver transformation may proceed using {BRONZE_FQN}.<table>_validated tables.")